In [ ]:
"""Cell 1: evidence-led APT candidates for Fe-Mn-C-Si-Al steels.

Run this file before cell2_gui_basic_advanced.py in JupyterLab.
The catalogue is deliberately finite: isotope variants are generated from a
selected published ion, not offered as separate ion-selection checkboxes.
"""

from __future__ import annotations

from html import escape
from itertools import product
from math import isfinite, log10
from pathlib import Path
import json
import os
import re
import tempfile


# NIST Atomic Weights and Isotopic Compositions; fractions are terrestrial
# natural values, not measured APT peak heights. 2H (D) is included.
ISOTOPES = {
    "H": ((1, 1.00782503223, 0.999885), (2, 2.01410177812, 0.000115)),
    "C": ((12, 12.0, 0.9893), (13, 13.00335483507, 0.0107)),
    "O": ((16, 15.99491461957, 0.99757), (17, 16.99913175650, 0.00038), (18, 17.99915961286, 0.00205)),
    "Al": ((27, 26.98153853, 1.0),),
    "Si": ((28, 27.97692653465, 0.92223), (29, 28.97649466490, 0.04685), (30, 29.973770136, 0.03092)),
    "Mn": ((55, 54.93804391, 1.0),),
    "Fe": ((54, 53.93960899, 0.05845), (56, 55.93493633, 0.91754), (57, 56.93539284, 0.02119), (58, 57.93327443, 0.00282)),
    "Ga": ((69, 68.9255735, 0.60108), (71, 70.92470258, 0.39892)),
}
ELECTRON_MASS_U = 0.0005485799090441  # NIST 2022 CODATA
ISOTOPE_SOURCE = "https://physics.nist.gov/cgi-bin/Compositions/stand_alone.pl"
ELECTRON_SOURCE = "https://physics.nist.gov/cgi-bin/cuu/Value?meu"

SOURCES = {
    "Wei 2021": "https://doi.org/10.1016/j.patter.2020.100192",
    "Toji 2014": "https://doi.org/10.1016/j.actamat.2013.10.064",
    "Toji 2016": "https://doi.org/10.1016/j.actamat.2016.06.044",
    "Kwok 2020": "https://doi.org/10.1016/j.msea.2020.139258",
    "Rementeria 2017": "https://doi.org/10.1016/j.actamat.2016.12.013",
    "Wang 2024": "https://doi.org/10.1038/s41529-024-00528-9",
    "Guerguis 2025": "https://doi.org/10.1093/mam/ozae126",
    "Dumas 2021": "https://doi.org/10.1016/j.ultramic.2020.113153",
    "Thuvander 2011": "https://doi.org/10.1016/j.ultramic.2010.12.024",
    "Marceau 2013": "https://doi.org/10.1016/j.ultramic.2013.01.010",
    "Niu 2024": "https://doi.org/10.1016/j.actamat.2024.120327",
}


def _ion(key, label, formula, charge, tab, group, source, locator, evidence,
         note=""):
    return {
        "key": key,
        "label": label,
        "formula": formula,
        "charge": charge,
        "tab": tab,
        "group": group,
        "source": source,
        "source_url": SOURCES[source],
        "locator": locator,
        "evidence": evidence,
        "note": note,
    }


# Display order is intentional. No ion is selected by default in Cell 2.
ION_CATALOG = [
    # BASIC — matrix and solute atomic ions.
    _ion("Fe2", "Fe²⁺", {"Fe": 1}, 2, "Basic", "Atomic ions", "Wei 2021", "Table 2", "reported", "⁵⁴Fe²⁺ can overlap Al⁺."),
    _ion("Fe1", "Fe⁺", {"Fe": 1}, 1, "Basic", "Atomic ions", "Wei 2021", "Table 2", "co-assigned", "Fe⁺ / FeH⁺ region is co-assigned."),
    _ion("Mn2", "Mn²⁺", {"Mn": 1}, 2, "Basic", "Atomic ions", "Toji 2014", "Section 2.3", "reported"),
    _ion("Mn1", "Mn⁺", {"Mn": 1}, 1, "Basic", "Atomic ions", "Wei 2021", "Table 2", "reported"),
    _ion("C1", "C⁺", {"C": 1}, 1, "Basic", "Atomic ions", "Wei 2021", "Table 2", "reported", "12 Da can also admit C₂²⁺ as an alternative."),
    _ion("C2charge", "C²⁺", {"C": 1}, 2, "Basic", "Atomic ions", "Wei 2021", "Table 2", "reported"),
    _ion("Si2", "Si²⁺", {"Si": 1}, 2, "Basic", "Atomic ions", "Toji 2014", "Section 2.3", "reported", "²⁸Si²⁺ can overlap ¹⁴N⁺."),
    _ion("Al2", "Al²⁺", {"Al": 1}, 2, "Basic", "Atomic ions", "Wei 2021", "Table 2", "reported"),
    _ion("Al1", "Al⁺", {"Al": 1}, 1, "Basic", "Atomic ions", "Kwok 2020", "APT mass-to-charge peak list", "overlap", "27 Da can overlap ⁵⁴Fe²⁺."),
    _ion("Al3", "Al³⁺", {"Al": 1}, 3, "Basic", "Atomic ions", "Wei 2021", "Table 2", "reported"),
    # BASIC — carbon clusters and the published mixed carbide ion.
    _ion("C2cluster1", "C₂⁺", {"C": 2}, 1, "Basic", "Carbon clusters", "Toji 2014", "Section 2.3", "overlap", "Some peaks overlap C₄²⁺ isotopologues."),
    _ion("C3cluster1", "C₃⁺", {"C": 3}, 1, "Basic", "Carbon clusters", "Toji 2014", "Section 2.3", "reported"),
    _ion("C3cluster2", "C₃²⁺", {"C": 3}, 2, "Basic", "Carbon clusters", "Wei 2021", "Table 2", "reported"),
    _ion("C4cluster1", "C₄⁺", {"C": 4}, 1, "Basic", "Carbon clusters", "Toji 2016", "APT mass-spectra paragraph", "reported"),
    _ion("C4cluster2", "C₄²⁺", {"C": 4}, 2, "Basic", "Carbon clusters", "Toji 2014", "Section 2.3", "overlap", "Some peaks overlap C₂⁺ isotopologues."),
    _ion("FeC2_2", "FeC₂²⁺", {"Fe": 1, "C": 2}, 2, "Basic", "Metal–carbon ion", "Kwok 2020", "APT mass-to-charge peak list", "overlap", "Wei 2021 also assigns it; other formulas can contribute near 40 Da."),
    _ion("FeC1", "FeC⁺", {"Fe": 1, "C": 1}, 1, "Basic", "Metal–carbon ion", "Guerguis 2025", "Figure 10(c)", "reported", "Published in an Fe capping-layer APT spectrum on an Fe–Mn specimen; not proof of occurrence in every steel."),
    _ion("FeC2_1", "FeC₂⁺", {"Fe": 1, "C": 2}, 1, "Basic", "Metal–carbon ion", "Guerguis 2025", "Figure 10(c)", "reported", "Published in an Fe capping-layer APT spectrum on an Fe–Mn specimen; not proof of occurrence in every steel."),
    _ion("SiC2_2", "SiC₂²⁺", {"Si": 1, "C": 2}, 2, "Basic", "Metal–carbon ion", "Thuvander 2011", "Table 3, SiC column", "reported", "Published in SiC; use as a literature-backed candidate, not a confirmed steel assignment."),
    _ion("SiC2_1", "SiC₂⁺", {"Si": 1, "C": 2}, 1, "Basic", "Metal–carbon ion", "Thuvander 2011", "Table 3, SiC column", "reported", "Published in SiC; use as a literature-backed candidate, not a confirmed steel assignment."),
    _ion("Si3C2", "Si₃C²⁺", {"Si": 3, "C": 1}, 2, "Basic", "Metal–carbon ion", "Dumas 2021", "Figure 3(c)", "reported", "Published in carbon-implanted silicon; not proof of occurrence in this steel."),
    # ADVANCED — other verified alloy charge states.
    _ion("Fe3", "Fe³⁺", {"Fe": 1}, 3, "Advanced", "Other reported alloy states", "Rementeria 2017", "Section 3.3", "reported", "Broader Fe–Mn–C–Si steel context."),
    _ion("Si1", "Si⁺", {"Si": 1}, 1, "Advanced", "Other reported alloy states", "Wang 2024", "Supplementary Table S1", "overlap", "Reported in pipeline steel; isotope region can overlap Fe²⁺."),
    # ADVANCED — diagnostic hydrogen, oxygen and preparation-related ions.
    _ion("H1", "H⁺", {"H": 1}, 1, "Advanced", "Hydrogen-related", "Wang 2024", "Hydrogen mass-spectrum discussion", "reported", "A 1 Da signal can originate from residual-gas background."),
    _ion("H2", "H₂⁺", {"H": 2}, 1, "Advanced", "Hydrogen-related", "Wei 2021", "Table 2", "reported", "Check background and preparation controls."),
    _ion("FeH1", "FeH⁺", {"Fe": 1, "H": 1}, 1, "Advanced", "Hydrogen-related", "Wei 2021", "Table 2", "co-assigned", "Co-assigned with Fe⁺ in the cited region."),
    _ion("O1", "O⁺", {"O": 1}, 1, "Advanced", "Oxygen-related", "Wei 2021", "Table 2", "reported"),
    _ion("OH1", "OH⁺", {"O": 1, "H": 1}, 1, "Advanced", "Oxygen-related", "Wei 2021", "Table 2 (HO⁺ notation)", "reported"),
    _ion("O2", "O₂⁺", {"O": 2}, 1, "Advanced", "Oxygen-related", "Wei 2021", "Table 2", "reported"),
    _ion("FeO2charge", "FeO²⁺", {"Fe": 1, "O": 1}, 2, "Advanced", "Oxygen-related", "Wei 2021", "Table 2", "reported"),
    _ion("FeO2_2charge", "FeO₂²⁺", {"Fe": 1, "O": 2}, 2, "Advanced", "Oxygen-related", "Wang 2024", "Figure 2 and accompanying mass-spectrum text", "reported", "Published in pipeline steel; oxide-layer context matters."),
    _ion("SiO1", "SiO⁺", {"Si": 1, "O": 1}, 1, "Advanced", "Oxygen-related", "Dumas 2021", "Figure 3(b)", "reported", "Published in carbon-implanted silicon; not proof of occurrence in this steel."),
    _ion("SiOH1", "SiOH⁺", {"Si": 1, "O": 1, "H": 1}, 1, "Advanced", "Oxygen-related", "Dumas 2021", "Figure 3(b)", "reported", "Published in carbon-implanted silicon; not proof of occurrence in this steel."),
    _ion("SiH2O2", "SiH₂O²⁺", {"Si": 1, "H": 2, "O": 1}, 2, "Advanced", "Oxygen-related", "Dumas 2021", "Figure 3(a)", "reported", "Published in carbon-implanted silicon; not proof of occurrence in this steel."),
    _ion("Ga1", "Ga⁺", {"Ga": 1}, 1, "Advanced", "FIB/preparation signal", "Marceau 2013", "APT mass spectra section", "reported", "Possible gallium implantation during FIB preparation."),
    _ion("Ga2", "Ga²⁺", {"Ga": 1}, 2, "Advanced", "FIB/preparation signal", "Marceau 2013", "APT mass spectra section", "reported", "Possible gallium implantation during FIB preparation."),
    # Explicitly not a verified default candidate.
    _ion("C2cluster2", "C₂²⁺", {"C": 2}, 2, "Advanced", "Tentative alternative", "Niu 2024", "Figure 2 and text", "tentative only", "Upper-bound alternative to C⁺ at 12 Da; not uniquely demonstrated."),
]
ION_BY_KEY = {item["key"]: item for item in ION_CATALOG}
ELEMENT_ORDER = ("Fe", "C", "Mn", "Si", "Al", "H", "O", "Ga")


def ion_is_available(ion, selected_elements):
    """Only display a formula when all its elements have been chosen."""
    return set(ion["formula"]).issubset(set(selected_elements))


def isotope_peaks(ion):
    """Enumerate every stable-isotope combination for one selected ion.

    Identical atoms are aggregated by isotopologue; their probabilities sum,
    so C₂ with one ¹³C is counted twice and remains one plotted peak.
    """
    atom_symbols = [symbol for symbol, count in ion["formula"].items()
                    for _ in range(count)]
    combined = {}
    for combination in product(*(ISOTOPES[s] for s in atom_symbols)):
        isotope_ids = tuple(sorted((symbol, item[0])
                                   for symbol, item in zip(atom_symbols, combination)))
        probability = 1.0
        for item in combination:
            probability *= item[2]
        if isotope_ids not in combined:
            combined[isotope_ids] = {
                "mass_u": sum(item[1] for item in combination),
                "probability": 0.0,
                "nominal_mass": sum(item[0] for item in combination),
            }
        combined[isotope_ids]["probability"] += probability

    peaks = []
    charge = ion["charge"]
    for isotope_ids, record in combined.items():
        isotope_label = "·".join(f"{mass_number}{symbol}"
                                 for symbol, mass_number in isotope_ids)
        exact_mz = (record["mass_u"] - charge * ELECTRON_MASS_U) / charge
        peaks.append({
            "ion_key": ion["key"],
            "ion": ion["label"],
            "isotopes": isotope_label,
            "charge": charge,
            "mass": exact_mz,
            "nominal_mz": record["nominal_mass"] / charge,
            "integer_mz": round(exact_mz),
            "abundance": 100.0 * record["probability"],
            "source_url": ion["source_url"],
            "evidence": ion["evidence"],
            "category": ion["group"],
            "family": ion["label"],
            "label": ion["label"],
        })
    return sorted(peaks, key=lambda peak: peak["mass"])


def principal_peak(ion):
    return max(isotope_peaks(ion), key=lambda peak: peak["abundance"])


def isotope_position_summary(ion):
    """Show every isotope combination with a highlighted nominal m/z.

    Keeping the isotope labels distinguishes combinations that share the same
    nominal position. Exact masses remain available in plots and exports.
    """
    return "; ".join(
        f"{escape(peak['isotopes'])}: <strong style='color:#15803D'>{peak['nominal_mz']:.1f}</strong>"
        for peak in isotope_peaks(ion)
    )


def selected_candidate_peaks(selected_keys):
    """Create all isotope lines for checked ion families, in catalogue order."""
    selected_keys = set(selected_keys)
    return [peak for ion in ION_CATALOG if ion["key"] in selected_keys
            for peak in isotope_peaks(ion)]


def parse_manual_candidates(text):
    """Read `ion name: m/z` entries separated by newlines or semicolons.

    Manual peaks are single positions supplied by the user, not literature
    assignments or generated isotope families.
    """
    candidates = []
    for entry in re.split(r"[;\n]+", text):
        entry = entry.strip()
        if not entry:
            continue
        if ":" not in entry:
            raise ValueError(f"Manual peak needs 'ion name: m/z': {entry}")
        label, mass_text = entry.rsplit(":", 1)
        label = label.strip()
        if not label:
            raise ValueError("A manual peak needs an ion name before the colon.")
        try:
            mass = float(mass_text.strip())
        except ValueError as error:
            raise ValueError(f"Invalid m/z for manual peak {label!r}.") from error
        if not isfinite(mass) or mass <= 0:
            raise ValueError(f"Manual m/z must be a positive finite number: {label!r}.")
        candidates.append({
            "ion_key": f"manual-{len(candidates)}",
            "ion": label,
            "label": label,
            "isotopes": "user-entered position",
            "charge": "",
            "mass": mass,
            "nominal_mz": None,
            "integer_mz": round(mass),
            "abundance": 100.0,
            "source_url": "",
            "evidence": "manual (unverified)",
            "category": "Manual peaks",
            "family": label,
        })
    return candidates


def output_paths_for_epos(epos_filename, folder_text=""):
    """Return named exports derived from the EPOS filename.

    Candidate ions, unknown peak positions, and the measured spectrum are
    separate files so their purpose is clear when stored beside the EPOS file.
    """
    epos_path = Path(epos_filename).expanduser()
    folder = Path(folder_text).expanduser() if folder_text.strip() else epos_path.parent
    if not folder.is_dir():
        raise FileNotFoundError(f"Save folder does not exist: {folder}")
    for number in range(1, 1001):
        suffix = "" if number == 1 else f"_{number}"
        stem = f"{epos_path.stem}{suffix}"
        html_path = folder / f"{stem}_mass-spectrum.html"
        txt_path = folder / f"{stem}_candidate-list.txt"
        csv_path = folder / f"{stem}_mass-spectrum.csv"
        if not any(path.exists() for path in (html_path, txt_path, csv_path)):
            return html_path, txt_path, csv_path
    raise FileExistsError("No unused output name found in the save folder.")


PROJECT_FORMAT = "apt-candidate-list-maker-project"
PROJECT_VERSION = 2


def default_project_path(epos_filename):
    """Suggest a project filename beside the EPOS file."""
    if not str(epos_filename).strip():
        raise ValueError("Enter an EPOS path or a project-file path first.")
    epos_path = Path(epos_filename).expanduser()
    return epos_path.with_name(f"{epos_path.stem}.apt-project.json")


def validate_project_state(state):
    """Validate a saved session without requiring APAV or a live EPOS file."""
    if not isinstance(state, dict) or state.get("format") != PROJECT_FORMAT:
        raise ValueError("This is not an APTpy — Spectrum Analyser project file.")
    if type(state.get("version")) is not int or state["version"] not in (1, PROJECT_VERSION):
        raise ValueError("Unsupported project-file version.")
    for name in ("selected_elements", "selected_ion_keys"):
        values = state.get(name)
        if not isinstance(values, list) or any(not isinstance(value, str) for value in values):
            raise ValueError(f"Project field {name!r} must be a list of names.")
        if len(values) != len(set(values)):
            raise ValueError(f"Project field {name!r} contains duplicates.")
    elements = set(state["selected_elements"])
    keys = set(state["selected_ion_keys"])
    if elements - set(ELEMENT_ORDER):
        raise ValueError(f"Unknown element(s) in project: {sorted(elements - set(ELEMENT_ORDER))}")
    if keys - set(ION_BY_KEY):
        raise ValueError(f"Unknown ion key(s) in project: {sorted(keys - set(ION_BY_KEY))}")
    if any(not ion_is_available(ION_BY_KEY[key], elements) for key in keys):
        raise ValueError("A selected ion requires an element not selected in this project.")
    for name in ("epos_file", "manual_peaks", "save_folder"):
        if not isinstance(state.get(name), str):
            raise ValueError(f"Project field {name!r} must be text.")
    for name in ("lower_mz", "upper_mz", "bin_width"):
        value = state.get(name)
        if type(value) not in (int, float) or not isfinite(value):
            raise ValueError(f"Project field {name!r} must be a finite number.")
    if state.get("y_scale") not in ("linear", "log"):
        raise ValueError("Project y-axis scale must be linear or log.")
    unknown_peaks = state.get("unknown_peaks", [])
    if not isinstance(unknown_peaks, list):
        raise ValueError("Project field 'unknown_peaks' must be a list.")
    for peak in unknown_peaks:
        if not isinstance(peak, dict) or not isinstance(peak.get("peak_id"), str):
            raise ValueError("Each unknown peak must have a text peak_id.")
        position = peak.get("position_da")
        if type(position) not in (int, float) or not isfinite(position) or position <= 0:
            raise ValueError("Each unknown peak position must be a positive finite number.")
    selected_ions = state.get("selected_ions", [])
    if not isinstance(selected_ions, list) or any(not isinstance(item, dict) for item in selected_ions):
        raise ValueError("Project field 'selected_ions' must be a list of ion records.")
    output_files = state.get("output_files", {})
    if not isinstance(output_files, dict) or any(not isinstance(key, str) or not isinstance(value, str) for key, value in output_files.items()):
        raise ValueError("Project field 'output_files' must be a text map.")
    spectrum_cache = state.get("spectrum_cache")
    if spectrum_cache is not None:
        if not isinstance(spectrum_cache, dict):
            raise ValueError("Project field 'spectrum_cache' must be an object.")
        for name in ("lower_mz", "upper_mz", "bin_width"):
            value = spectrum_cache.get(name)
            if type(value) not in (int, float) or not isfinite(value):
                raise ValueError(f"Cached spectrum field {name!r} must be a finite number.")
        mass = spectrum_cache.get("mass")
        counts = spectrum_cache.get("counts")
        if (not isinstance(mass, list) or not isinstance(counts, list) or
                len(mass) != len(counts) or not mass):
            raise ValueError("Cached spectrum mass and counts must be non-empty lists of equal length.")
        if any(type(value) not in (int, float) or not isfinite(value) for value in mass + counts):
            raise ValueError("Cached spectrum values must be finite numbers.")
    return state


def save_project_file(filename, state, *, overwrite=False):
    """Write a validated JSON project atomically; never silently replace one."""
    state = validate_project_state(state)
    path = Path(filename).expanduser()
    if path.suffix.lower() != ".json":
        raise ValueError("Project filename must end in .json.")
    if not path.parent.is_dir():
        raise FileNotFoundError(f"Project folder does not exist: {path.parent}")
    if path.exists() and not overwrite:
        raise FileExistsError(f"Project already exists: {path}. Load it, or choose a new filename.")
    if overwrite and path.exists():
        existing = json.loads(path.read_text(encoding="utf-8"))
        if not isinstance(existing, dict):
            raise ValueError("The existing project must be a JSON object.")
        state = {**existing, **state}
    temporary_name = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="w", encoding="utf-8", prefix=f".{path.name}.",
            suffix=".tmp", dir=path.parent, delete=False,
        ) as output:
            temporary_name = output.name
            json.dump(state, output, ensure_ascii=False, indent=2)
            output.write("\n")
            output.flush()
            os.fsync(output.fileno())
        if overwrite:
            os.replace(temporary_name, path)
        else:
            os.link(temporary_name, path)
    finally:
        if temporary_name is not None:
            Path(temporary_name).unlink(missing_ok=True)
    return path


def load_project_file(filename):
    """Read and validate a JSON project before changing any GUI controls."""
    path = Path(filename).expanduser()
    with path.open("r", encoding="utf-8") as project_file:
        return validate_project_state(json.load(project_file))


def spectrum_histogram(roi, lower_mass, upper_mass, bin_width):
    """Return exactly the bins and counts used by the displayed spectrum."""
    mass, counts = roi.mass_histogram(
        bin_width=bin_width, lower=lower_mass, upper=upper_mass,
        multiplicity="all",
    )
    mass, counts = list(mass), list(counts)
    if len(mass) != len(counts):
        raise ValueError("APAV returned different lengths for m/z and counts.")
    return mass, counts


def spectrum_cache_path(epos_filename):
    """Return the reusable mass-spectrum cache beside an EPOS file."""
    path = Path(epos_filename).expanduser()
    return path.with_name(f"{path.stem}.apt-spectrum-cache.json")


def validate_spectrum_cache(cache):
    """Validate a standalone cached histogram before it is used for plotting."""
    if not isinstance(cache, dict):
        raise ValueError("Spectrum cache must be an object.")
    for name in ("epos_file", "lower_mz", "upper_mz", "bin_width"):
        if name == "epos_file":
            if not isinstance(cache.get(name), str):
                raise ValueError("Spectrum cache epos_file must be text.")
        elif type(cache.get(name)) not in (int, float) or not isfinite(cache[name]):
            raise ValueError(f"Spectrum cache field {name!r} must be finite.")
    mass, counts = cache.get("mass"), cache.get("counts")
    if (not isinstance(mass, list) or not isinstance(counts, list) or
            not mass or len(mass) != len(counts)):
        raise ValueError("Spectrum cache mass and counts must be equal non-empty lists.")
    if any(type(value) not in (int, float) or not isfinite(value)
           for value in mass + counts):
        raise ValueError("Spectrum cache values must be finite numbers.")
    return cache


def load_spectrum_cache_file(filename):
    path = Path(filename).expanduser()
    with path.open("r", encoding="utf-8") as cache_file:
        return validate_spectrum_cache(json.load(cache_file))


def save_spectrum_cache_file(filename, cache):
    path = Path(filename).expanduser()
    path.write_text(json.dumps(validate_spectrum_cache(cache), ensure_ascii=False),
                    encoding="utf-8")
    return path


def load_spectrum_csv(filename):
    """Read the exported two-column mass-spectrum CSV."""
    import csv
    mass, counts = [], []
    with Path(filename).expanduser().open("r", encoding="utf-8", newline="") as source:
        reader = csv.DictReader(source)
        for row in reader:
            mass.append(float(row["m/z (Da/e)"]))
            counts.append(float(row["Counts"]))
    if not mass or len(mass) != len(counts):
        raise ValueError("Mass-spectrum CSV is empty or malformed.")
    if any(not isfinite(value) for value in mass + counts):
        raise ValueError("Mass-spectrum CSV contains non-finite values.")
    return mass, counts


PALETTE = (
    "#2166AC", "#D73027", "#1B7837", "#762A83", "#E08214", "#008B8B",
    "#8C510A", "#C51B7D", "#4D4D4D", "#7FBC41", "#A6761D", "#1F78B4",
    "#B15928", "#6A3D9A", "#E31A1C", "#33A02C", "#FB9A99", "#CAB2D6",
    "#B2DF8A", "#FDBF6F", "#006D2C", "#08519C", "#A50F15", "#54278F",
)
DASHES = ("solid", "dash", "dot", "dashdot")


def _style_for(ion_key):
    if ion_key.startswith("manual-"):
        index = len(ION_CATALOG) + int(ion_key.split("-", 1)[1])
    else:
        index = next(i for i, ion in enumerate(ION_CATALOG) if ion["key"] == ion_key)
    return {
        "color": PALETTE[index % len(PALETTE)],
        "dash": DASHES[index // len(PALETTE) % len(DASHES)],
    }


def make_spectrum_plot(roi, candidates, lower_mass=0.0, upper_mass=200.0,
                       bin_width=0.01, logarithmic_y=False, show_labels=False,
                       spectrum=None):
    """Plot a cached or freshly calculated EPOS histogram and ion guides."""
    import plotly.graph_objects as go

    mass, counts = (spectrum if spectrum is not None else
                    spectrum_histogram(roi, lower_mass, upper_mass, bin_width))
    if not mass or not counts or not any(value > 0 for value in counts):
        raise ValueError("No positive spectrum counts in this m/z interval.")
    positive = [value for value in counts if value > 0]
    y_max = float(max(positive))
    y_min = max(1.0, float(min(positive)))
    y_top_limit = y_max * 1.5 if y_max > 1 else 10.0
    fig = go.Figure()
    fig.add_trace(go.Scattergl(
        x=mass, y=counts, mode="lines", name="Measured spectrum",
        line={"color": "#24292F", "width": 1}, showlegend=False,
        hovertemplate="m/z %{x:.5f}<br>Counts %{y:.0f}<extra></extra>",
    ))

    family_peaks = {}
    for candidate in candidates:
        if lower_mass <= candidate["mass"] <= upper_mass:
            family_peaks.setdefault(candidate["ion_key"], []).append(candidate)

    plotted_ions = list(ION_CATALOG)
    for key, peaks in family_peaks.items():
        if key.startswith("manual-"):
            plotted_ions.append({"key": key, "label": peaks[0]["ion"],
                                 "evidence": "manual (unverified)",
                                 "manual": True})

    def family_abundance_max(ion):
        if ion.get("manual"):
            return max(peak["abundance"] for peak in family_peaks[ion["key"]])
        return max(peak["abundance"] for peak in isotope_peaks(ion))

    # Candidate heights are proportional to isotope abundance in data units.
    # Plotly then applies the requested axis transform. Do not interpolate the
    # abundance fraction across the logarithmic screen height: that makes a
    # log plot's guides behave like a linear-abundance slider.
    reference_height = 0.84 * y_top_limit
    visible_tops = []
    for ion in plotted_ions:
        peaks = family_peaks.get(ion["key"], [])
        if peaks:
            largest = family_abundance_max(ion)
            visible_tops.extend(reference_height * p["abundance"] / largest
                                for p in peaks)
    # A logarithmic counts axis must start at one count. Candidate isotope
    # guides are clipped to this floor instead of expanding the axis below 1.
    log_floor = (1.0 if logarithmic_y else
                 (max(1e-12, min(y_min, min(visible_tops) * 0.5))
                  if visible_tops else y_min))

    for ion in plotted_ions:
        peaks = family_peaks.get(ion["key"], [])
        if not peaks:
            continue
        peak_max = family_abundance_max(ion)
        style = _style_for(ion["key"])
        tallest = max(peaks, key=lambda peak: peak["abundance"])
        for peak in peaks:
            fraction = peak["abundance"] / peak_max
            line_base = log_floor if logarithmic_y else 0.0
            line_top = reference_height * fraction
            hover_text = (
                f"<b>{escape(ion['label'])}</b><br>"
                f"User-entered m/z: {peak['mass']:.6f} Da/e<br>"
                "Manual peak; not literature-verified<extra></extra>"
                if ion.get("manual") else
                f"<b>{escape(ion['label'])}</b><br>"
                f"Isotopes: {escape(peak['isotopes'])}<br>"
                f"Calculated m/z: {peak['mass']:.6f} Da/e<br>"
                f"Natural isotope share: {peak['abundance']:.5f}%<br>"
                f"Evidence: {escape(ion['evidence'])}<extra></extra>"
            )
            fig.add_trace(go.Scatter(
                x=[peak["mass"], peak["mass"]],
                y=[line_base, line_top], mode="lines", showlegend=False,
                line={"color": style["color"], "width": 2,
                      "dash": style["dash"]},
                hovertemplate=hover_text,
            ))
            if show_labels and peak is tallest:
                fig.add_annotation(
                    x=peak["mass"], y=line_top, text=escape(ion["label"]),
                    showarrow=False, yshift=8, font={"size": 10,
                    "color": style["color"]},
                )

    fig.update_layout(
        title="APT mass spectrum and selected candidate positions",
        template="plotly_white", height=750, hovermode="closest",
        showlegend=False, margin={"l": 75, "r": 40, "t": 75, "b": 80},
        dragmode="pan",
    )
    fig.update_xaxes(title="Mass-to-charge ratio (Da/e)",
                     range=[lower_mass, upper_mass], rangeslider_visible=True)
    fig.update_yaxes(title="Counts", type="log" if logarithmic_y else "linear",
                     range=[log10(log_floor), log10(max(y_top_limit, log_floor * 10))]
                     if logarithmic_y else [0, y_top_limit])
    return fig


In [ ]:
"""Cell 2: two-tab beginner interface for the published-ion catalogue.

Run cell1_core_basic_advanced.py first in the same Jupyter kernel.
Requires ipywidgets, plotly and APAV for EPOS plotting.
"""

from html import escape
from pathlib import Path
import csv
import io
import json
import subprocess


import ipywidgets as widgets
from IPython.display import Markdown, display

if "ION_CATALOG" not in globals():
    raise RuntimeError("Run cell1_core_basic_advanced.py first in this notebook.")


# No elements and no ions are selected by default.
element_widgets = {
    symbol: widgets.Checkbox(value=False, description=symbol, indent=False,
                             layout=widgets.Layout(width="70px"))
    for symbol in ELEMENT_ORDER
}
ion_widgets = {
    ion["key"]: widgets.Checkbox(value=False, description=ion["label"],
                                 indent=False,
                                 layout=widgets.Layout(width="100px"))
    for ion in ION_CATALOG
}
ion_rows = {}
group_rows = {}

for ion in ION_CATALOG:
    principal = principal_peak(ion)
    isotope_positions = isotope_position_summary(ion)
    source_url = escape(ion["source_url"], quote=True)
    source_name = escape(ion["source"])
    badge = " <span style='color:#B45309'>[tentative]</span>" if ion["evidence"] == "tentative only" else (
        " <span style='color:#A44336'>[overlap]</span>" if ion["evidence"] in ("overlap", "co-assigned") else ""
    )
    detail = widgets.HTML(
        value=(
            f"<span style='font-size:12px'>main <b>{principal['mass']:.6f}</b> Da/e; "
            f"isotopes (nominal m/z, Da/e): {isotope_positions}{badge}<br>"
            f"<a href='{source_url}' target='_blank'>{source_name}</a>"
            f" — {escape(ion['note'])}</span>"
        ),
        layout=widgets.Layout(width="calc(100% - 110px)"),
    )
    row = widgets.HBox(
        [ion_widgets[ion["key"]], detail],
        layout=widgets.Layout(width="100%", align_items="flex-start",
                              padding="6px 4px", border_bottom="1px solid #E5E7EB"),
    )
    ion_rows[ion["key"]] = row
    group_rows.setdefault((ion["tab"], ion["group"]), []).append(row)


def make_accordion(tab_name, ordered_groups):
    children = [widgets.VBox(group_rows[(tab_name, group)],
                             layout=widgets.Layout(width="100%"))
                for group in ordered_groups]
    accordion = widgets.Accordion(children=children,
                                  layout=widgets.Layout(width="100%"))
    for index, group in enumerate(ordered_groups):
        accordion.set_title(index, group)
    accordion.selected_index = 0 if tab_name == "Basic" else None
    return accordion


basic_accordion = make_accordion(
    "Basic", ("Atomic ions", "Carbon clusters", "Metal–carbon ion")
)
advanced_accordion = make_accordion(
    "Advanced", ("Other reported alloy states", "Hydrogen-related",
                 "Oxygen-related", "FIB/preparation signal",
                 "Tentative alternative")
)

def _selected_elements():
    return {symbol for symbol, control in element_widgets.items() if control.value}


def _selected_ions():
    elements = _selected_elements()
    return [ion for ion in ION_CATALOG
            if ion_widgets[ion["key"]].value and ion_is_available(ion, elements)]


def refresh_ion_rows(_=None):
    elements = _selected_elements()
    for ion in ION_CATALOG:
        available = ion_is_available(ion, elements)
        ion_rows[ion["key"]].layout.display = "" if available else "none"
        if not available and ion_widgets[ion["key"]].value:
            ion_widgets[ion["key"]].value = False


for control in element_widgets.values():
    control.observe(refresh_ion_rows, names="value")
for control in ion_widgets.values():
    control.observe(refresh_ion_rows, names="value")


epos_widget = widgets.Text(
    value="", description="EPOS file:",
    placeholder="Paste the full path to your .epos or .EPOS file",
    layout=widgets.Layout(width="100%"),
)
lower_widget = widgets.FloatText(value=0.0, description="Lower m/z:")
upper_widget = widgets.FloatText(value=200.0, description="Upper m/z:")
bin_widget = widgets.FloatText(value=0.01, description="Bin width:")
scale_widget = widgets.ToggleButtons(
    options=[("Linear", "linear"), ("Logarithmic", "log")],
    value="log", description="Y-axis:",
)
manual_widget = widgets.Textarea(
    value="",
    placeholder="Peak 1: 34; Peak 2: 36 (One peak per line or separated by semicolons)",
    layout=widgets.Layout(width="100%", height="78px"),
)
copy_unknown_button = widgets.Button(description="Copy peak list", icon="copy")
unknown_select_widget = (
    widgets.Dropdown(options=(), description="Selected peak:")
    if hasattr(widgets, "Dropdown") else
    widgets.Text(value="", description="Selected peak ID:")
)
delete_unknown_button = widgets.Button(description="Delete selected peak", icon="trash")
unknown_status = widgets.HTML(value="No unknown peaks saved.")
unknown_table_widget = widgets.HTML(value="")
unknown_peaks = []
_interactive_figure = None
save_folder_widget = widgets.Text(
    value="", description="Save folder:",
    placeholder="All files use the EPOS filename; blank saves beside EPOS, or enter another folder.",
    layout=widgets.Layout(width="100%"),
)
project_file_widget = widgets.Text(
    value="", description="Project file:",
    placeholder="Save/resume project (.json stores unknown peaks and cached spectrum): paste a path to load; blank saves beside EPOS.",
    layout=widgets.Layout(width="100%"),
)
save_project_button = widgets.Button(description="Save project", icon="save",
                                     button_style="success")
load_project_button = widgets.Button(description="Load project", icon="folder-open")
project_status = widgets.HTML(value="Load a project. Changes save automatically.")
project_path_display = widgets.HTML(value="")
save_folder_display = widgets.HTML(value="")
choose_save_folder_button = widgets.Button(description="Choose save folder", icon="folder-open")
blank_button = widgets.Button(description="Plot spectrum only",
                              button_style="info", icon="line-chart")
candidate_button = widgets.Button(description="Plot selected ions",
                                  button_style="success", icon="line-chart")
export_button = widgets.Button(description="Save plot, spectrum and ion list",
                               button_style="primary", icon="download")
status_output = widgets.Output()
plot_output = widgets.Output(layout=widgets.Layout(width="100%"))

_cached_roi = None
_cached_signature = None
_cached_mass = None
_cached_counts = None
_cached_spectrum_signature = None
_last_plot_mode = None
_active_project_path = None
_applying_project = False
_project_output_paths = {}


def _project_message(message, *, error=False):
    color = "#B42318" if error else "#176B45"
    rendered = escape(message).replace("\n", "<br>")
    project_status.value = f"<span style='color:{color}'>{rendered}</span>"


def _project_path():
    path_text = project_file_widget.value.strip()
    return (Path(path_text).expanduser() if path_text else
            default_project_path(epos_widget.value.strip())).absolute()


def _project_state():
    return {
        "format": PROJECT_FORMAT,
        "version": PROJECT_VERSION,
        "epos_file": epos_widget.value,
        "selected_elements": [symbol for symbol, control in element_widgets.items()
                              if control.value],
        "selected_ion_keys": [ion["key"] for ion in ION_CATALOG
                              if ion_widgets[ion["key"]].value],
        "selected_ions": [
            {
                "key": ion["key"], "label": ion["label"],
                "formula": dict(ion["formula"]), "charge": int(ion["charge"]),
                "tab": ion["tab"], "group": ion["group"],
                "source": ion["source"], "source_url": ion["source_url"],
                "locator": ion["locator"], "evidence": ion["evidence"],
                "note": ion["note"],
                "isotopes": [
                    {"isotopes": peak["isotopes"],
                     "charge": int(peak["charge"]),
                     "exact_mz": float(peak["mass"]),
                     "nominal_mz": float(peak["nominal_mz"]),
                     "nearest_integer_mz": int(peak["integer_mz"]),
                     "natural_isotope_share_percent": float(peak["abundance"])}
                    for peak in isotope_peaks(ion)
                ],
            }
            for ion in ION_CATALOG if ion_widgets[ion["key"]].value
        ],
        "manual_peaks": manual_widget.value,
        "lower_mz": lower_widget.value,
        "upper_mz": upper_widget.value,
        "bin_width": bin_widget.value,
        "y_scale": scale_widget.value,
        "save_folder": save_folder_widget.value,
        "output_files": dict(_project_output_paths),
        "unknown_peaks": [{"peak_id": str(peak["peak_id"]),
                           "position_da": float(peak["position_da"])}
                          for peak in unknown_peaks],
        "spectrum_cache": (
            {"lower_mz": float(_cached_spectrum_signature[1]),
             "upper_mz": float(_cached_spectrum_signature[2]),
             "bin_width": float(_cached_spectrum_signature[3]),
             "mass": [float(value) for value in _cached_mass],
             "counts": [float(value) for value in _cached_counts]}
            if _cached_spectrum_signature is not None and
               _cached_mass is not None and _cached_counts is not None
            else None
        ),
    }


def _save_project(_):
    global _active_project_path, _applying_project
    try:
        # Saving a project with an EPOS path also snapshots its spectrum, so
        # future sessions can work without reopening the EPOS file.
        path = _project_path()
        if (epos_widget.value.strip() or _project_output_paths or
                _cached_mass is not None):
            if epos_widget.value.strip():
                _spectrum_for_current_settings()
            saved_exports = _write_project_exports()
        else:
            saved_exports = {}
        save_project_file(path, _project_state(),
                          overwrite=(path == _active_project_path))
        _active_project_path = path
        _applying_project = True
        try:
            project_file_widget.value = str(path)
        finally:
            _applying_project = False
        messages = [f"Project saved: {path}."]
        labels = {
            "candidate_list_txt": "Candidate list saved",
            "unknown_peaks_txt": "Unknown peak positions saved",
            "mass_spectrum_csv": "Mass spectrum CSV saved",
            "mass_spectrum_html": "Mass spectrum HTML saved",
        }
        messages.extend(f"{labels.get(key, 'Output saved')}: {value}"
                        for key, value in saved_exports.items())
        _project_message("\n".join(messages))
    except Exception as error:
        _project_message(f"{type(error).__name__}: {error}", error=True)


def _load_project(_):
    global _active_project_path, _applying_project
    global _last_plot_mode, _cached_mass, _cached_counts, _cached_spectrum_signature, _project_output_paths
    try:
        path = _project_path()
        state = load_project_file(path)
        _last_plot_mode = None  # Loading must not redraw a previous project's plot.
        _project_output_paths = {str(key): str(value) for key, value in state.get("output_files", {}).items()}
        if not _project_output_paths and state.get("epos_file"):
            try:
                html_path, txt_path, csv_path = output_paths_for_epos(
                    state["epos_file"], state.get("save_folder", "")
                )
                _project_output_paths = {
                    "candidate_list_txt": str(txt_path),
                    "unknown_peaks_txt": str(txt_path.with_name(txt_path.stem.replace("_candidate-list", "_unknown-peak-positions") + ".txt")),
                    "mass_spectrum_csv": str(csv_path),
                    "mass_spectrum_html": str(html_path),
                }
            except (OSError, ValueError):
                _project_output_paths = {}
        cached = state.get("spectrum_cache")
        if cached is None and state.get("epos_file"):
            try:
                cached = load_spectrum_cache_file(spectrum_cache_path(state["epos_file"]))
            except (OSError, ValueError, json.JSONDecodeError):
                cached = None
        if cached is None:
            _cached_mass = _cached_counts = _cached_spectrum_signature = None
        else:
            _cached_mass = [float(value) for value in cached["mass"]]
            _cached_counts = [float(value) for value in cached["counts"]]
            _cached_spectrum_signature = (
                str(cached.get("epos_file", state["epos_file"])),
                float(cached["lower_mz"]), float(cached["upper_mz"]),
                float(cached["bin_width"])
            )
        _applying_project = True
        try:
            for control in ion_widgets.values():
                control.value = False
            for symbol, control in element_widgets.items():
                control.value = symbol in state["selected_elements"]
            for key in state["selected_ion_keys"]:
                ion_widgets[key].value = True
            # A cached project is self-contained; leave the original EPOS
            # address blank so a fresh session does not depend on that file.
            epos_widget.value = "" if cached is not None else state["epos_file"]
            manual_widget.value = state["manual_peaks"]
            lower_widget.value = state["lower_mz"]
            upper_widget.value = state["upper_mz"]
            bin_widget.value = state["bin_width"]
            scale_widget.value = state["y_scale"]
            save_folder_widget.value = state["save_folder"]
            unknown_peaks[:] = [{"peak_id": peak["peak_id"],
                                 "position_da": float(peak["position_da"]) }
                                for peak in state.get("unknown_peaks", [])]
            project_file_widget.value = str(path)
            refresh_ion_rows()
            _render_unknown_table()
        finally:
            _applying_project = False
        _active_project_path = path
        project_path_display.value = "Project file: " + escape(str(path))
        folder = Path(save_folder_widget.value).expanduser() if save_folder_widget.value.strip() else path.parent
        save_folder_display.value = "Save folder: " + escape(str(folder.absolute()))
        with plot_output:
            plot_output.clear_output()
        cache_message = (" Cached mass spectrum loaded; the original EPOS file is not required."
                         if cached is not None
                         else " No cached spectrum is present; the first plot requires the EPOS file.")
        _project_message(f"Project loaded: {path}." + cache_message)
    except Exception as error:
        _project_message(f"{type(error).__name__}: {error}", error=True)


def _project_file_changed(change):
    global _active_project_path
    if _applying_project or _active_project_path is None:
        return
    if change["new"].strip() != str(_active_project_path):
        _active_project_path = None
        _project_message("Project path changed. Click Save project or Load project.")





def _settings():
    if lower_widget.value >= upper_widget.value:
        raise ValueError("Lower m/z must be smaller than upper m/z.")
    if bin_widget.value <= 0:
        raise ValueError("Bin width must be positive.")


def _load_roi():
    global _cached_roi, _cached_signature
    path_text = epos_widget.value.strip()
    if not path_text:
        raise ValueError("Enter the full path to your EPOS file.")
    path = Path(path_text).expanduser()
    if not path.is_file():
        raise FileNotFoundError(f"EPOS file not found: {path}")
    stat = path.stat()
    signature = (str(path.resolve()), stat.st_mtime_ns, stat.st_size)
    if signature != _cached_signature:
        import apav as ap
        _cached_roi = ap.Roi.from_epos(str(path))
        _cached_signature = signature
    return _cached_roi


def _spectrum_for_current_settings():
    """Use the saved CSV first, then cache, before opening EPOS."""
    global _cached_mass, _cached_counts, _cached_spectrum_signature
    epos_text = epos_widget.value.strip()
    signature = (epos_text, float(lower_widget.value), float(upper_widget.value),
                 float(bin_widget.value))
    cache_matches = (_cached_spectrum_signature is not None and
                     _cached_spectrum_signature[1:] == signature[1:] and
                     (not epos_text or _cached_spectrum_signature[0] == epos_text))
    if cache_matches and _cached_mass is not None and _cached_counts is not None:
        return list(_cached_mass), list(_cached_counts)
    csv_path_text = _project_output_paths.get("mass_spectrum_csv", "")
    if csv_path_text:
        try:
            csv_mass, csv_counts = load_spectrum_csv(csv_path_text)
            _cached_mass, _cached_counts = csv_mass, csv_counts
            _cached_spectrum_signature = signature
            return list(_cached_mass), list(_cached_counts)
        except (OSError, ValueError, KeyError):
            pass
    cache_path = spectrum_cache_path(epos_text) if epos_text else None
    if cache_path is not None and cache_path.is_file():
        try:
            cached = load_spectrum_cache_file(cache_path)
            cached_signature = (cached["epos_file"], float(cached["lower_mz"]),
                                float(cached["upper_mz"]), float(cached["bin_width"]))
            if cached_signature == signature:
                _cached_mass = [float(value) for value in cached["mass"]]
                _cached_counts = [float(value) for value in cached["counts"]]
                _cached_spectrum_signature = signature
                return list(_cached_mass), list(_cached_counts)
        except (OSError, ValueError, json.JSONDecodeError):
            pass
    mass, counts = spectrum_histogram(_load_roi(), lower_widget.value,
                                      upper_widget.value, bin_widget.value)
    _cached_mass = [float(value) for value in mass]
    _cached_counts = [float(value) for value in counts]
    _cached_spectrum_signature = signature
    if cache_path is not None:
        save_spectrum_cache_file(cache_path, {
            "epos_file": epos_text, "lower_mz": signature[1],
            "upper_mz": signature[2], "bin_width": signature[3],
            "mass": _cached_mass, "counts": _cached_counts,
        })
    return list(_cached_mass), list(_cached_counts)


def _build_figure(with_candidates):
    _settings()
    selected_ions = _selected_ions() if with_candidates else []
    manual_peaks = parse_manual_candidates(manual_widget.value) if with_candidates else []
    if with_candidates and not selected_ions and not manual_peaks:
        raise ValueError("Select an ion or enter a manual peak, or use 'Plot spectrum only'.")
    spectrum = _spectrum_for_current_settings()
    roi = None
    candidates = selected_candidate_peaks(ion["key"] for ion in selected_ions)
    candidates.extend(manual_peaks)
    figure = make_spectrum_plot(
        roi=roi,
        candidates=candidates,
        lower_mass=lower_widget.value,
        upper_mass=upper_widget.value,
        bin_width=bin_widget.value,
        logarithmic_y=scale_widget.value == "log",
        show_labels=False,
        spectrum=spectrum,
    )
    return figure, selected_ions, candidates


def _unknown_mass_counts():
    return _spectrum_for_current_settings()


def _persist_active_project():
    """Update an already-open project after peaks or spectrum cache change."""
    global _active_project_path
    if _applying_project:
        return
    path = _active_project_path
    if path is None:
        if not epos_widget.value.strip():
            _project_message("Load a project to enable automatic saving.")
            return
        path = _project_path()
        if path.exists():
            raise FileExistsError("A project already exists here. Load it to resume automatic saving.")
    save_project_file(path, _project_state(), overwrite=(path == _active_project_path))
    _active_project_path = path
    project_file_widget.value = str(path)
    project_path_display.value = "Project file: " + escape(str(path))
    _project_message("All changes saved automatically.")


def _add_unknown_peak(position):
    try:
        position = float(position)
        if position <= 0:
            raise ValueError("Peak position must be positive.")
        record = {"peak_id": f"Peak {len(unknown_peaks) + 1}",
                  "position_da": position}
        unknown_peaks.append(record)
        _save_unknown_peaks()
        _render_unknown_table()
        unknown_status.value = (f"<span style='color:#176B45'>Peak {len(unknown_peaks)} added at {position:.6f} Da/e.</span>")
    except Exception as error:
        unknown_status.value = f"<span style='color:#B42318'>{escape(str(error))}</span>"


def _add_manual_peaks(_):
    try:
        entries = parse_manual_candidates(manual_widget.value)
        if not entries:
            raise ValueError("Enter at least one manual peak, for example: Peak 1: 34")
        for entry in entries:
            unknown_peaks.append({"peak_id": f"Peak {len(unknown_peaks) + 1}",
                                  "position_da": float(entry["mass"])})
        for index, peak in enumerate(unknown_peaks, start=1):
            peak["peak_id"] = f"Peak {index}"
        _save_unknown_peaks()
        _render_unknown_table()
        unknown_status.value = (f"<span style='color:#176B45'>{len(entries)} manual peak position(s) added.</span>")
        manual_widget.value = ""
    except Exception as error:
        unknown_status.value = f"<span style='color:#B42318'>{escape(str(error))}</span>"


def _delete_unknown_peak(_):
    selected = unknown_select_widget.value
    if not selected:
        unknown_status.value = "Select a peak before deleting it."
        return
    unknown_peaks[:] = [p for p in unknown_peaks if p["peak_id"] != selected]
    for index, peak in enumerate(unknown_peaks, start=1):
        peak["peak_id"] = f"Peak {index}"
    _save_unknown_peaks()
    _render_unknown_table()
    unknown_status.value = "Selected peak deleted."


def _copy_unknown_peaks(_):
    if not unknown_peaks:
        unknown_status.value = "There are no peak positions to copy."
        return
    text = "Peak ID\tPosition (Da/e)\n" + "\n".join(
        f"{p['peak_id']}\t{p['position_da']:.8f}" for p in unknown_peaks
    ) + "\n"
    try:
        subprocess.run(["pbcopy"], input=text, text=True, check=True)
        unknown_status.value = f"<span style='color:#176B45'>Copied {len(unknown_peaks)} peak position(s).</span>"
    except Exception as error:
        unknown_status.value = f"<span style='color:#B42318'>Could not copy peak list: {escape(str(error))}</span>"


def _unknown_list_path():
    saved = _project_output_paths.get("unknown_peaks_txt")
    if saved:
        return Path(saved).expanduser()
    if _active_project_path is not None:
        name = _active_project_path.name.removesuffix(".apt-project.json").removesuffix(".json")
        folder = Path(save_folder_widget.value).expanduser() if save_folder_widget.value.strip() else _active_project_path.parent
        return folder / (name + "_unknown-peak-positions.txt")
    epos_text = epos_widget.value.strip()
    if not epos_text:
        raise ValueError("Load a project before adding an unknown peak.")
    epos_path = Path(epos_text).expanduser()
    folder = Path(save_folder_widget.value).expanduser() if save_folder_widget.value.strip() else epos_path.parent
    return folder / (epos_path.stem + "_unknown-peak-positions.txt")


def _save_unknown_peaks():
    _persist_active_project()
    path = _unknown_list_path()
    lines = ["Peak ID\tPosition (Da/e)"]
    lines.extend(f"{p['peak_id']}\t{p['position_da']:.8f}" for p in unknown_peaks)
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")


def _render_unknown_table():
    if not unknown_peaks:
        unknown_table_widget.value = ""
        unknown_select_widget.options = ()
        return
    unknown_select_widget.options = tuple((f"{p['peak_id']} — {p['position_da']:.6f} Da/e", p["peak_id"])
                                         for p in unknown_peaks)
    rows = "".join(
        f"<tr><td>{escape(p['peak_id'])}</td><td>{p['position_da']:.6f}</td></tr>"
        for p in unknown_peaks
    )
    unknown_table_widget.value = (
        "<table style='border-collapse:collapse'><tr><th>Peak</th><th>Position (Da/e)</th></tr>" + rows + "</table>"
    )


copy_unknown_button.on_click(_copy_unknown_peaks)
delete_unknown_button.on_click(_delete_unknown_peak)


def _plot(with_candidates):
    global _last_plot_mode, _interactive_figure
    with status_output:
        status_output.clear_output()
        try:
            figure, selected_ions, candidates = _build_figure(with_candidates)
            figure.update_layout(autosize=True, width=None)
            with plot_output:
                plot_output.clear_output()
                # FigureWidget sends the clicked x-coordinate back to this
                # notebook. This is the preferred way to mark an unknown peak;
                # the fallback keeps plotting usable when anywidget is absent.
                try:
                    import plotly.graph_objects as go
                    _interactive_figure = go.FigureWidget(figure)
                    _interactive_figure._config = {
                        "scrollZoom": True,
                        "displaylogo": False,
                        "responsive": True,
                    }
                    if _interactive_figure.data:
                        def _capture_click(trace, points, selector):
                            if points.xs:
                                _add_unknown_peak(float(points.xs[0]))
                        _interactive_figure.data[0].on_click(_capture_click)
                    display(_interactive_figure)
                except Exception:
                    figure.show(config={"scrollZoom": True, "displaylogo": False,
                                        "responsive": True})
            _last_plot_mode = with_candidates
            _persist_active_project()
            manual_count = sum(peak["evidence"].startswith("manual") for peak in candidates)
            print(f"Plotted {len(selected_ions)} ion families, {manual_count} manual peaks, "
                  f"and {len(candidates)} candidate lines.")
        except Exception as error:
            print(f"{type(error).__name__}: {error}")


blank_button.on_click(lambda _: _plot(False))
candidate_button.on_click(lambda _: _plot(True))


def _refresh_scale(change):
    # A toggle changes the displayed guide heights immediately; no second
    # click on the plot button is needed. The EPOS data are reused from cache.
    if change["name"] == "value" and _last_plot_mode is not None:
        _plot(_last_plot_mode)


scale_widget.observe(_refresh_scale, names="value")


ACKNOWLEDGMENT = """## Acknowledgment and citation

If this software contributes to a publication, presentation, thesis, or other research output, please acknowledge the author and cite the software release.

Suggested acknowledgment:
The authors acknowledge Roohallah Surki Aliabad for developing APTpy — Spectrum Analyser, a publicly available JupyterLab tool for candidate-ion generation and mass-spectrum visualization based on APAV.

Suggested software citation:

> Aliabad, R. S. (2026). *APTpy — Spectrum Analyser* (Version 1.0.1) [Computer software]. Zenodo. [https://doi.org/10.5281/zenodo.23296361](https://doi.org/10.5281/zenodo.23296361)

Please also cite APAV separately when it is used in the analysis:
Smith, J. D., & Young, M. L. (2023). APAV: An Open-Source Python Package for Mass Spectrum Analysis in Atom Probe Tomography. Journal of Open Source Software, 8(83), 4862. https://doi.org/10.21105/joss.04862
"""


def _write_project_exports():
    """Save candidate-list, unknown-peak, and mass-spectrum files."""
    global _project_output_paths
    epos_text = epos_widget.value.strip()
    has_saved_paths = all(_project_output_paths.get(key) for key in (
        "candidate_list_txt", "unknown_peaks_txt", "mass_spectrum_csv", "mass_spectrum_html"))
    if not epos_text and not has_saved_paths:
        project_path = _project_path()
        project_name = project_path.name
        suffix = ".apt-project.json"
        stem = project_name[:-len(suffix)] if project_name.endswith(suffix) else project_path.stem
        folder = Path(save_folder_widget.value).expanduser() if save_folder_widget.value.strip() else project_path.parent
        html_path = folder / f"{stem}_mass-spectrum.html"
        txt_path = folder / f"{stem}_candidate-list.txt"
        csv_path = folder / f"{stem}_mass-spectrum.csv"
    else:
        html_path = txt_path = csv_path = None
    if save_folder_widget.value.strip() and has_saved_paths:
        output_folder = Path(save_folder_widget.value).expanduser()
        html_path = output_folder / Path(_project_output_paths["mass_spectrum_html"]).name
        txt_path = output_folder / Path(_project_output_paths["candidate_list_txt"]).name
        csv_path = output_folder / Path(_project_output_paths["mass_spectrum_csv"]).name
    _settings()
    selected_ions = _selected_ions()
    use_candidates = bool(selected_ions or manual_widget.value.strip())
    figure, _, peaks = _build_figure(use_candidates)
    if html_path is None:
        if has_saved_paths:
            html_path = Path(_project_output_paths["mass_spectrum_html"])
            txt_path = Path(_project_output_paths["candidate_list_txt"])
            csv_path = Path(_project_output_paths["mass_spectrum_csv"])
        else:
            html_path, txt_path, csv_path = output_paths_for_epos(epos_text, save_folder_widget.value)
    mass, counts = _spectrum_for_current_settings()
    buffer = io.StringIO(); writer = csv.writer(buffer, delimiter="\t", lineterminator="\n")
    writer.writerow(["Ion", "Isotopes", "Charge", "m/z (Da/e)", "Nearest integer", "Natural isotope share (%)", "Evidence", "Source", "Source URL"])
    for peak in peaks:
        ion = ION_BY_KEY.get(peak["ion_key"])
        writer.writerow([peak["ion"], peak["isotopes"], peak["charge"], f"{peak['mass']:.8f}", peak["integer_mz"], f"{peak['abundance']:.8f}" if ion else "", ion["evidence"] if ion else "manual (unverified)", ion["source"] if ion else "user-entered", ion["source_url"] if ion else ""])
    spectrum_buffer = io.StringIO(); spectrum_writer = csv.writer(spectrum_buffer, lineterminator="\n")
    spectrum_writer.writerow(["m/z (Da/e)", "Counts"]); spectrum_writer.writerows(zip(mass, counts))
    figure.write_html(str(html_path), include_plotlyjs=True, full_html=True)
    txt_path.write_text(buffer.getvalue(), encoding="utf-8"); csv_path.write_text(spectrum_buffer.getvalue(), encoding="utf-8")
    unknown_path = txt_path.with_name(txt_path.stem.replace("_candidate-list", "_unknown-peak-positions") + ".txt")
    unknown_buffer = io.StringIO(); unknown_writer = csv.writer(unknown_buffer, delimiter="\t", lineterminator="\n")
    unknown_writer.writerow(["Peak ID", "Position (Da/e)"])
    for unknown in unknown_peaks: unknown_writer.writerow([unknown["peak_id"], f"{unknown['position_da']:.8f}"])
    unknown_path.write_text(unknown_buffer.getvalue(), encoding="utf-8")
    _project_output_paths = {"candidate_list_txt": str(txt_path), "unknown_peaks_txt": str(unknown_path), "mass_spectrum_csv": str(csv_path), "mass_spectrum_html": str(html_path)}
    return _project_output_paths


def export_selected_ions(_):
    with status_output:
        status_output.clear_output()
        try:
            for path in _write_project_exports().values(): print(path)
        except Exception as error:
            print(f"{type(error).__name__}: {error}")



basic_tab = widgets.VBox([
    widgets.HTML("<b>Basic:</b> reported core-steel ions, ordered by element, then carbon clusters and FeC₂²⁺. Overlap notes do not hide candidates."),
    basic_accordion,
])
advanced_tab = widgets.VBox([
    widgets.HTML("<b>Advanced:</b> other published states, H/O/Ga diagnostics, and one explicitly tentative alternative. Isotopes are not separate checkboxes."),
    advanced_accordion,
])
tabs = widgets.Tab(children=[basic_tab, advanced_tab],
                   layout=widgets.Layout(width="100%"))
tabs.set_title(0, "Basic ions")
tabs.set_title(1, "Advanced ions")


import asyncio
import subprocess
_autosave_handle = None


def _run_autosave():
    global _autosave_handle
    _autosave_handle = None
    try:
        _persist_active_project()
        return True
    except Exception as error:
        _project_message("Automatic save failed: " + str(error), error=True)
        return False


def _queue_autosave(change=None):
    global _autosave_handle
    if _applying_project:
        return
    if _autosave_handle is not None:
        _autosave_handle.cancel()
    _project_message("Saving changes…")
    try:
        _autosave_handle = asyncio.get_running_loop().call_later(0.35, _run_autosave)
    except RuntimeError:
        _run_autosave()


def _flush_autosave():
    global _autosave_handle
    if _autosave_handle is not None:
        _autosave_handle.cancel()
        _autosave_handle = None
        return _run_autosave()
    return True


def _choose_project(_):
    if not _flush_autosave():
        return
    try:
        result = subprocess.run(["osascript", "-e", 'POSIX path of (choose file with prompt "Load an APTpy project JSON")'], capture_output=True, text=True)
        if result.returncode:
            return
        path = Path(result.stdout.strip()).resolve()
        load_project_file(path)  # Validate before changing the active project.
        project_file_widget.value = str(path)
        _load_project(None)
    except Exception as error:
        _project_message(str(error), error=True)


def _choose_save_folder(_):
    try:
        result = subprocess.run(["osascript", "-e", 'POSIX path of (choose folder with prompt "Choose the save folder")'], capture_output=True, text=True)
        if result.returncode:
            return
        folder = Path(result.stdout.strip()).resolve()
        if not folder.is_dir():
            raise ValueError("Choose an existing folder.")
        save_folder_widget.value = str(folder)
        save_folder_display.value = "Save folder: " + escape(str(folder))
        for key, filename in list(_project_output_paths.items()):
            _project_output_paths[key] = str(folder / Path(filename).name)
        _queue_autosave()
    except Exception as error:
        _project_message(str(error), error=True)


load_project_button.on_click(_choose_project)
choose_save_folder_button.on_click(_choose_save_folder)
for control in [*element_widgets.values(), *ion_widgets.values(), manual_widget,
                lower_widget, upper_widget, bin_widget, scale_widget]:
    control.observe(_queue_autosave, names="value")


interface = widgets.VBox([
    widgets.HTML("<h3>APTpy — Candidate List Maker</h3>"),
    widgets.HBox([load_project_button, choose_save_folder_button]),
    project_path_display,
    save_folder_display,
    project_status,
    widgets.VBox([
        widgets.HTML("<b>1 · Plot spectrum</b><br>Load the project data and set the spectrum display range."),
        epos_widget,
        widgets.HBox([lower_widget, upper_widget, bin_widget]),
        scale_widget,
        blank_button,
    ], layout=widgets.Layout(width="100%", padding="10px", background_color="#EAF7FA")),
    widgets.VBox([
        widgets.HTML("<b>2 · Ion family selection</b><br>Choose elements, then check the ions you want to plot."),
        widgets.HBox(list(element_widgets.values())),
        tabs,
    ], layout=widgets.Layout(width="100%", padding="10px", background_color="#F3F4F6")),
    widgets.VBox([
        widgets.HTML("<b>3 · Unknown peak positions (Da/e)</b>"),
        widgets.HTML("<b style='color:#B42318'>Click the top of an unassigned peak to add it automatically.</b>"),
        widgets.HBox([copy_unknown_button]),
        unknown_status,
        widgets.HBox([unknown_select_widget, delete_unknown_button]),
        widgets.HTML("<b>3.1 · Add peak positions manually</b><br>Enter positions above, then click <i>Plot selected ions</i>."),
        manual_widget,
    ], layout=widgets.Layout(width="100%", padding="10px", background_color="#EAF7FA")),
    widgets.Box([candidate_button], layout=widgets.Layout(width="100%", justify_content="flex-start", padding="6px 0")),
    status_output,
    plot_output,
], layout=widgets.Layout(width="100%"))

refresh_ion_rows()
display(interface)
